# Downloading data

In [27]:
import numpy as np
import pandas as pd
import yfinance as yf

In [28]:
ticker = "AAPL"
start_date = "2010-01-01"
end_date = "2024-12-31"

In [29]:
df = yf.download(
        tickers=ticker,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=True,
        progress=False
    )

In [30]:
df = df.copy().sort_index()
df.columns = df.columns.droplevel('Ticker')

In [31]:
def transformation(data):
        # YFinance indexes by Date automatically; ensure it is sorted
        data = data.sort_index()
        
        # Calculate log-returns: r_t = ln(P_t / P_{t-1}) [cite: 5353]
        ohlc_cols = ['Open', 'High', 'Low', 'Close']
        # Divide OHLC by previous day's close to get relative returns
        prev_close = data['Close'].shift(1)
        ohlc_log_rets = np.log(data[ohlc_cols].div(prev_close, axis=0))
        
        # Volume log-returns (adding 1 to avoid log(0))
        volume_log_rets = np.log(data['Volume'] + 1) - np.log(data['Volume'].shift(1) + 1)

        processed_data = pd.concat([ohlc_log_rets, volume_log_rets], axis=1).dropna()
        data_arr = processed_data.values.astype(np.float32)

        # Global Standardization: r_std = (r - mu) / sigma [cite: 5370]
        mean = data_arr.mean(axis=0)
        std = data_arr.std(axis=0)
        data_standardized = (data_arr - mean) / (std + 1e-8)
        
        return data_standardized

In [32]:
data = transformation(df)

In [33]:
print(data)

[[ 0.18371652 -0.23282808  0.37985665  0.04303514  0.63514984]
 [-0.05353456 -0.48656994 -0.4974031  -0.968573   -0.27335787]
 [ 0.26442647 -0.41840288  0.01744855 -0.16076277 -0.4643574 ]
 ...
 [-0.05685903 -0.23464318  0.46727428  0.12522142  0.5099844 ]
 [-0.4502784  -0.8742328  -0.89974266 -0.8145871   1.4127659 ]
 [-1.1937099  -1.3948922  -0.62979066 -0.81581    -0.556674  ]]


In [ ]:
file_path = f'../data/{ticker}_{start_date}_{end_date}_processed.csv'

# 3. Save to CSV
# We keep index=True to save the Date column
columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# 2. Convert the NumPy array back to a pandas DataFrame
df_processed = pd.DataFrame(data, columns=columns)

df_processed.to_csv(file_path, index=False)

print(f"Data saved successfully to {file_path}")

Data saved successfully to ../datasets/AAPL_2010-01-01_2024-12-31_processed.csv
